# Experiment: MCMC-IS Beta Diagnostics

Objective:
- Fix one exact scenario.
- Compare MCMC-IS performance across beta values.
- Record estimates and diagnostics at intermediate estimation points.

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "perm_pval").exists() and (candidate / "results").exists():
            return candidate
    raise RuntimeError("Could not locate project root containing perm_pval/ and results/.")


project_root = find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from perm_pval.experiments.notebook_studies import (
    BetaSweepStudyConfig,
    CrossMethodStudyConfig,
    MCMCWorkflowConfig,
    SAMCWorkflowConfig,
    build_beta_workflow,
    create_timestamped_run_dir,
    load_beta_sweep_saved_output,
    load_cross_method_saved_output,
    load_selected_scenarios,
    regenerate_beta_sweep_plots_from_saved,
    regenerate_cross_method_plots_from_saved,
    run_beta_checkpoint_study,
    run_cross_method_study,
    save_beta_sweep_outputs,
    save_cross_method_outputs,
)

pd.set_option("display.max_columns", 100)
project_root

PosixPath('/Users/noamchowers/Documents/University/Thesis/Code/MCMCIS')

## Configuration

`ESTIMATION_POINTS` controls checkpoint budgets.  
The final checkpoint is used for the main beta boxplots; all checkpoints are used for convergence plots.

In [ ]:
FAST_MODE = False
SAVE_OUTPUTS = True

CATALOG_PATH = project_root / "results" / "exact_scenarios" / "v1" / "catalog.json"
OUTPUT_ROOT = project_root / "results" / "mcmcis_beta_notebook"

SCENARIO_KEYS_TO_RUN = [
    "hypergeom_1e7",
    "gwas_additive_score_n40",
    "linear_stat_dp_n40",
    "bruteforce_welch_nonextreme_n22",
]
ESTIMATION_POINTS = (333_000, 1_000_000, 2_500_000, 5_000_000, 10_000_000) if not FAST_MODE else (2_000, 10_000, 20_000)
BETA_MULTIPLIERS = (0.5, 0.75, 1.00, 1.25, 1.5)
BETA_REPEATS = 5 if not FAST_MODE else 2
N_JOBS = min(BETA_REPEATS, os.cpu_count() or 1)
MIN_TAIL_STATES = 2
BASE_SEED = 54_321

mcmc_cfg = MCMCWorkflowConfig(
    pilot_samples=20_000 if not FAST_MODE else 1_000,
    tune_steps=2_000 if not FAST_MODE else 1_000,
    local_scan_screen_total_steps=12_000 if not FAST_MODE else 1_000,
    local_scan_total_steps=64_000 if not FAST_MODE else 6_000,
    chains=2,
    thin=1,
    estimate_variance=True,
    proposal_size=0.1,
)
beta_cfg = BetaSweepStudyConfig(
    estimation_points=ESTIMATION_POINTS,
    repeats=BETA_REPEATS,
    beta_multipliers=BETA_MULTIPLIERS,
    chains=2,
    thin=1,
    estimate_variance=True,
    proposal_size=0.075,
    base_seed=BASE_SEED,
    n_jobs=N_JOBS,
)

print(json.dumps({
    "FAST_MODE": FAST_MODE,
    "SCENARIO_KEYS_TO_RUN": SCENARIO_KEYS_TO_RUN,
    "ESTIMATION_POINTS": ESTIMATION_POINTS,
    "BETA_MULTIPLIERS": BETA_MULTIPLIERS,
    "BETA_REPEATS": BETA_REPEATS,
    "N_JOBS": N_JOBS,
    "SAVE_OUTPUTS": SAVE_OUTPUTS,
}, indent=2))


{
  "FAST_MODE": false,
  "SCENARIO_KEYS_TO_RUN": [
    "hypergeom_1e7",
    "gwas_additive_score_n40",
    "linear_stat_dp_n40",
    "bruteforce_welch_nonextreme_n22"
  ],
  "ESTIMATION_POINTS": [
    333000,
    1000000,
    2500000,
    5000000,
    10000000
  ],
  "BETA_MULTIPLIERS": [
    0.5,
    0.75,
    1.0,
    1.25,
    1.5
  ],
  "BETA_REPEATS": 5,
  "N_JOBS": 5,
  "SAVE_OUTPUTS": true
}


## Load Scenario And Build Beta Workflow

In [3]:
scenarios = load_selected_scenarios(
    catalog_path=CATALOG_PATH,
    scenario_keys=SCENARIO_KEYS_TO_RUN,
    min_tail_states=MIN_TAIL_STATES,
)

run_dir = create_timestamped_run_dir(OUTPUT_ROOT, "beta_diag") if SAVE_OUTPUTS else None

pd.DataFrame([
    {
        "scenario": s.key,
        "exact_p": s.exact_p,
        "tail_hits": s.exact_tail_hits,
        "n_perm": s.exact_n_perm,
        "exact_method": s.exact_method,
    }
    for s in scenarios
])


,scenario,exact_p,tail_hits,n_perm,exact_method
0,hypergeom_1e7,3.854286e-07,53130,137846528820,Fisher exact test (2x2; hypergeometric tail)
1,gwas_additive_score_n40,9.121811e-07,125741,137846528820,LinearStatisticDPSolver
2,linear_stat_dp_n40,8.124978e-09,1120,137846528820,LinearStatisticDPSolver
3,bruteforce_welch_nonextreme_n22,1.559328e-05,11,705432,BruteForceExactSolver


## Run Beta Sweep Across Checkpoints

In [ ]:
beta_results = {}

for scenario_idx, scenario in enumerate(scenarios):
    workflow_seed = BASE_SEED + 10_000 + 1_000 * scenario_idx
    beta_workflow = build_beta_workflow(
        scenario.problem,
        scenario.exact_p,
        mcmc_cfg,
        seed=workflow_seed,
    )

    print(json.dumps({
        "scenario": scenario.key,
        "exact_p": scenario.exact_p,
        "beta0_laplace": beta_workflow["beta0_laplace"],
        "beta_hat_tuned": beta_workflow["beta_hat_tuned"],
        "beta_used": beta_workflow["beta_used"],
        "q_target": beta_workflow["q_target"],
    }, indent=2))

    beta_study = run_beta_checkpoint_study(
        scenario.problem,
        scenario.exact_p,
        beta_center=float(beta_workflow["beta_used"]),
        sigma_t=float(beta_workflow["sigma_t"]),
        beta_cfg=beta_cfg,
    )
    beta_results[scenario.key] = {
        "study": beta_study,
        "beta_workflow": beta_workflow,
    }

    if SAVE_OUTPUTS and run_dir is not None:
        save_beta_sweep_outputs(
            beta_study,
            output_dir=run_dir / scenario.key,
            scenario_name=scenario.description,
            exact_p=scenario.exact_p,
            beta_cfg=beta_cfg,
            beta_workflow=beta_workflow,
        )

    beta_summary_df = pd.DataFrame(beta_study["summary"]).sort_values(["checkpoint", "beta"])
    display(beta_summary_df[[
        "checkpoint",
        "beta",
        "mean_estimate",
        "rmse",
        "mean_variance_estimate",
        "mean_q_tilt_tail_share",
        "mean_ess",
        "mean_acceptance_rate",
        "mean_weight_cv",
    ]])


{
  "scenario": "hypergeom_1e7",
  "exact_p": 3.8542863904376695e-07,
  "beta0_laplace": 3.396484375,
  "beta_hat_tuned": 3.6278367042541504,
  "beta_used": 2.720877528190613,
  "q_target": 0.024916440223237257
}


,checkpoint,beta,mean_estimate,rmse,mean_variance_estimate,mean_q_tilt_tail_share,mean_ess,mean_acceptance_rate,mean_weight_cv
0,333000,1.360439,2.531549e-07,1.387611e-07,1.712952e-14,0.000073,4.008598e+04,0.700022,2.417625
1,333000,2.040658,3.565884e-07,9.273474e-08,6.636972e-15,0.000882,5.595678e+03,0.565416,7.216759
2,333000,2.720878,3.858831e-07,4.843747e-08,4.630186e-15,0.005959,9.086108e+02,0.451849,23.944969
3,333000,3.401097,4.266974e-07,6.295524e-08,7.870632e-15,0.022230,1.788899e+02,0.355498,40.138778
4,333000,4.081316,6.114760e-07,2.334727e-07,1.770853e-14,0.066187,1.286045e+02,0.278368,49.585902
5,1000000,1.360439,3.413893e-07,1.004674e-07,8.529240e-15,0.000099,1.286165e+05,0.699907,2.285516
6,1000000,2.040658,3.790360e-07,4.001653e-08,2.534191e-15,0.000960,1.621460e+04,0.566734,6.967827
7,1000000,2.720878,3.837370e-07,3.009049e-08,1.275505e-15,0.005724,1.739772e+03,0.451308,23.506181
8,1000000,3.401097,4.283337e-07,7.428941e-08,4.486846e-15,0.022463,3.913434e+02,0.355230,59.196619
9,1000000,4.081316,5.116527e-07,1.552802e-07,1.620672e-14,0.067031,1.469602e+02,0.277511,118.113620


{
  "scenario": "gwas_additive_score_n40",
  "exact_p": 9.121811124035818e-07,
  "beta0_laplace": 3.650390625,
  "beta_hat_tuned": 4.22076416015625,
  "beta_used": 3.1655731201171875,
  "q_target": 0.030904396624839247
}


,checkpoint,beta,mean_estimate,rmse,mean_variance_estimate,mean_q_tilt_tail_share,mean_ess,mean_acceptance_rate,mean_weight_cv
0,333000,1.582787,7.248683e-07,2.442372e-07,4.563548e-14,0.000247,23468.604512,0.654843,3.232404
1,333000,2.374180,9.433555e-07,8.012591e-08,2.034668e-14,0.002543,1996.699808,0.515818,12.782455
2,333000,3.165573,8.725732e-07,9.741033e-08,3.674802e-14,0.012217,138.424031,0.402076,44.666366
3,333000,3.956966,1.269148e-06,4.096143e-07,6.373611e-14,0.042056,138.269916,0.314674,52.018560
4,333000,4.748360,1.597033e-06,9.633777e-07,2.732548e-13,0.111305,70.319524,0.245759,100.120459
5,1000000,1.582787,7.852830e-07,1.828316e-07,1.814941e-14,0.000265,66189.208277,0.654447,3.335136
6,1000000,2.374180,9.036932e-07,3.135807e-08,6.690007e-15,0.002405,6602.308891,0.514784,11.204227
7,1000000,3.165573,9.002457e-07,8.782093e-08,1.079901e-14,0.012412,459.333683,0.402554,46.694411
8,1000000,3.956966,1.180162e-06,3.510331e-07,2.649297e-14,0.041831,333.032902,0.313929,58.025527
9,1000000,4.748360,1.801670e-06,9.297145e-07,1.298145e-13,0.108309,208.238910,0.245938,76.707545


{
  "scenario": "linear_stat_dp_n40",
  "exact_p": 8.124977898155825e-09,
  "beta0_laplace": 4.830078125,
  "beta_hat_tuned": 4.815337896347046,
  "beta_used": 3.6115034222602844,
  "q_target": 0.009494138154016053
}


,checkpoint,beta,mean_estimate,rmse,mean_variance_estimate,mean_q_tilt_tail_share,mean_ess,mean_acceptance_rate,mean_weight_cv
0,333000,1.805752,3.602953e-09,6.606605e-09,9.055003e-17,0.000005,12976.036380,0.584791,4.442880
1,333000,2.708628,9.566108e-09,2.719929e-09,2.722328e-17,0.000139,1104.976044,0.426417,19.906800
2,333000,3.611503,1.075544e-08,5.602214e-09,1.221818e-17,0.000830,276.904635,0.306777,59.622642
3,333000,4.514379,1.237447e-08,5.552580e-09,2.178617e-17,0.003721,95.357308,0.221551,67.085811
4,333000,5.417255,2.863708e-08,2.209318e-08,1.147154e-16,0.010521,53.695373,0.161400,78.487796
5,1000000,1.805752,5.659810e-09,4.931222e-09,2.450420e-17,0.000007,36156.149322,0.584621,4.624822
6,1000000,2.708628,7.825660e-09,2.148916e-09,7.695974e-18,0.000118,1704.551524,0.426869,22.230091
7,1000000,3.611503,8.210488e-09,2.294509e-09,3.981647e-18,0.000810,302.214317,0.307000,79.232331
8,1000000,4.514379,1.162523e-08,4.000261e-09,7.596084e-18,0.003293,137.332764,0.221665,96.084487
9,1000000,5.417255,2.602904e-08,2.092150e-08,4.255840e-17,0.010767,125.456744,0.161818,90.599207


{
  "scenario": "bruteforce_welch_nonextreme_n22",
  "exact_p": 1.5593281847151816e-05,
  "beta0_laplace": 1.69873046875,
  "beta_hat_tuned": 1.7650871276855469,
  "beta_used": 1.3238153457641602,
  "q_target": 0.06283974039419676
}


## Review Saved Figures

In [ ]:
if SAVE_OUTPUTS and run_dir is not None:
    print(f"Saved outputs under: {run_dir}")
    for scenario in scenarios:
        scenario_dir = run_dir / scenario.key
        print(f"\n{scenario.key}")
        display(Image(filename=str(scenario_dir / "beta_max_budget.png")))
        display(Image(filename=str(scenario_dir / "beta_convergence.png")))
else:
    print("SAVE_OUTPUTS=False, so no saved figures to display.")


## Reload Saved Results Without Rerunning

In [ ]:
# RELOAD_BETA_DIR = None
# # Example:
# # RELOAD_BETA_DIR = project_root / "results" / "mcmcis_beta_notebook" / "20260306_120000_beta_diag" / "gwas_additive_score_n40"

# if RELOAD_BETA_DIR is not None:
#     saved = load_beta_sweep_saved_output(RELOAD_BETA_DIR)
#     print(json.dumps({
#         "scenario_display": saved["metadata"]["scenario_display"],
#         "exact_p": saved["metadata"]["exact_p"],
#         "beta_center": saved["metadata"]["settings"]["beta_center"],
#     }, indent=2))
#     regen = regenerate_beta_sweep_plots_from_saved(RELOAD_BETA_DIR)
#     for name, path in regen.items():
#         print(name, path)
#         display(Image(filename=str(path)))
# else:
#     print("Set RELOAD_BETA_DIR to a saved beta-study directory to regenerate plots from disk only.")
